# TabKAN (ChebyshevKAN) — Telco Churn

TabKAN (ChebyshevKAN) é uma rede KAN baseada em polinômios de Chebyshev, com suporte a pré-treino, fine-tuning (padrão ou GRPO), tuning de hiperparâmetros (Optuna) e interpretabilidade via importância de features.


In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from copy import deepcopy
from sklearn.model_selection import train_test_split
from model_utils import load_data, compute_metrics, save_results, print_scores


## 0. Instalar TabKAN

In [2]:
from tabkan import ChebyshevKAN
import tabkan

## 1. Carregar dados

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

Xtr_np  = X_train.values.astype('float32');  ytr_np  = y_train.values.astype(int)
Xvl_np  = X_val.values.astype('float32');    yvl_np  = y_val.values.astype(int)
Xte_np  = X_test.values.astype('float32');   yte_np  = y_test.values.astype(int)

n_features = Xtr_np.shape[1]
n_classes = len(np.unique(ytr_np))
print(f'n_features={n_features}  n_classes={n_classes}')


Train: (7242, 23)  Val: (1057, 23)  Test: (1057, 23)
n_features=23  n_classes=2


## 2. Preparar dataset no formato TabKAN

TabKAN espera um dicionário com `train_input`, `train_label`, `test_input`, `test_label` (tensores torch). Aqui usamos o Val como conjunto de "test" interno (para tuning/fit), mantendo o Test verdadeiro apenas para a avaliação final na seção 6.


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

def to_tensor_dataset(X, y, X_eval, y_eval, device=device):
    return {
        "train_input": torch.tensor(X, dtype=torch.float32).to(device),
        "train_label": torch.tensor(y, dtype=torch.long).to(device),
        "test_input":  torch.tensor(X_eval, dtype=torch.float32).to(device),
        "test_label":  torch.tensor(y_eval, dtype=torch.long).to(device),
    }

fit_dataset = to_tensor_dataset(Xtr_np, ytr_np, Xvl_np, yvl_np)

eval_dataset = to_tensor_dataset(Xtr_np, ytr_np, Xte_np, yte_np)

print(f'Train: {fit_dataset["train_input"].shape[0]} amostras (dataset completo, sem subsample)')
print(f'Churn rate treino: {ytr_np.mean():.3f}')


device: cuda
Train: 7242 amostras (dataset completo, sem subsample)
Churn rate treino: 0.500


## 3. Tuning de hiperparâmetros (Optuna)

In [5]:
search_space = {
    "depth": {"type": "int", "low": 1, "high": 2},
    "neurons_layer_0": {"type": "int", "low": 16, "high": 32},
    "neurons_layer_1": {"type": "int", "low": 8, "high": 16},
    "orders_layer_0": {"type": "int", "low": 2, "high": 4},
    "orders_layer_1": {"type": "int", "low": 2, "high": 4},
    "lr": {"type": "float", "low": 1e-2, "high": 1.0, "log": True},
    "steps": {"type": "categorical", "choices": [20]}
}

best_params = ChebyshevKAN.tune(
    model_class=ChebyshevKAN,
    dataset=fit_dataset,
    search_space=search_space,
    n_trials=5,
    device=device
)
print('Best hyperparameters found:', best_params)


[I 2026-07-01 20:35:28,920] A new study created in memory with name: no-name-2cc1d76c-4aca-4843-9c9c-93796230058f


  0%|          | 0/5 [00:00<?, ?it/s]

Training with L-BFGS:   0%|                                                  | 0/20 [00:00<?, ?it/s]

| train_loss: 6.89e-01 | test_loss: 7.05e-01 :   0%|                         | 0/20 [00:00<?, ?it/s]

| train_loss: 6.89e-01 | test_loss: 7.05e-01 :   5%|▊                | 1/20 [00:00<00:06,  3.09it/s]

...

| train_loss: 3.39e-01 | test_loss: 1.50e+00 :  90%|██████████████▍ | 18/20 [00:02<00:00,  8.45it/s]

| train_loss: 3.39e-01 | test_loss: 1.50e+00 :  95%|███████████████▏| 19/20 [00:02<00:00,  8.36it/s]

| train_loss: 3.30e-01 | test_loss: 1.53e+00 :  95%|███████████████▏| 19/20 [00:02<00:00,  8.36it/s]

| train_loss: 3.30e-01 | test_loss: 1.53e+00 : 100%|████████████████| 20/20 [00:02<00:00,  8.37it/s]

[I 2026-07-01 20:35:36,627] Trial 4 finished with value: 0.6725999958700725 and parameters: {'depth': 2, 'neurons_layer_0': 32, 'neurons_layer_1': 14, 'orders_layer_0': 4, 'orders_layer_1': 4, 'lr': 0.08139848856823335, 'steps': 20}. Best is trial 0 with value: 0.724953905689167.
Best hyperparameters found: {'depth': 1, 'neurons_layer_0': 27, 'orders_layer_0': 4, 'lr': 0.2950791304730668, 'steps': 20}


## 4. Fit (dataset completo)

In [6]:
params = dict(best_params)
depth = params.pop("depth")
lr = params.pop("lr")
steps = params.pop("steps")

layers = [n_features] + [params[f'neurons_layer_{i}'] for i in range(depth)] + [n_classes]
orders = [params[f'orders_layer_{i}'] for i in range(depth)]

clf = ChebyshevKAN(layers=layers, orders=orders).to(device)

clf.fit(
    fit_dataset,
    steps=steps,
    loss_fn=nn.CrossEntropyLoss(),
    lr=lr
)
print('TabKAN (ChebyshevKAN) fit concluído')


Training with L-BFGS:   0%|                                                  | 0/20 [00:00<?, ?it/s]

| train_loss: 6.89e-01 | test_loss: 7.06e-01 :   0%|                         | 0/20 [00:00<?, ?it/s]

| train_loss: 6.88e-01 | test_loss: 7.05e-01 :   0%|                         | 0/20 [00:00<?, ?it/s]

| train_loss: 6.88e-01 | test_loss: 7.05e-01 :  10%|█▋               | 2/20 [00:00<00:00, 18.07it/s]

| train_loss: 6.87e-01 | test_loss: 7.05e-01 :  10%|█▋               | 2/20 [00:00<00:00, 18.07it/s]

...

| train_loss: 6.86e-01 | test_loss: 7.04e-01 :  85%|█████████████▌  | 17/20 [00:00<00:00, 32.73it/s]

| train_loss: 6.86e-01 | test_loss: 7.04e-01 :  85%|█████████████▌  | 17/20 [00:00<00:00, 32.73it/s]

| train_loss: 6.86e-01 | test_loss: 7.04e-01 :  85%|█████████████▌  | 17/20 [00:00<00:00, 32.73it/s]

| train_loss: 6.86e-01 | test_loss: 7.04e-01 :  85%|█████████████▌  | 17/20 [00:00<00:00, 32.73it/s]

TabKAN (ChebyshevKAN) fit concluído


## 5. Predict (conjunto de teste)

In [7]:
clf.eval()
with torch.no_grad():
    logits = clf(eval_dataset['test_input'])
    probs_t = torch.softmax(logits, dim=1)[:, 1]
    probs = probs_t.cpu().numpy()
    preds = (probs >= 0.5).astype(int)

print(f'Predictions geradas: {len(preds)} amostras de teste')


Predictions geradas: 1057 amostras de teste


## 6. Avaliar no teste + salvar artefatos

In [8]:
scores = compute_metrics(yte_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/tabkan', exist_ok=True)
joblib.dump(clf, 'results/tabkan/model.pkl')

save_results('tabkan', yte_np, preds, probs, scores)
print(f'\nNota: TabKAN (ChebyshevKAN), treino completo com {len(Xtr_np)} amostras.')


Scores no teste:
  accuracy               0.7483
  balanced_accuracy      0.7706
  precision              0.5158
  recall                 0.8179
  f1                     0.6326
  auroc                  0.8549
  ks                     0.5520


Saved → results/tabkan

Nota: TabKAN (ChebyshevKAN), treino completo com 7242 amostras.
